In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Employee Attrition Big Data Project") \
    .getOrCreate()

spark

In [2]:
spark.version

'3.1.1'

In [4]:
# Reading the Data from Hadoop 

input_path = "hdfs://namenode:8020/employee_attrition/raw/Employee_Attrition_data.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(input_path)

In [5]:
df.show(5, truncate=False)

+---+---------+-----------------+---------+----------------------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+---------------------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|BusinessTravel   |DailyRate|Department            |DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|JobRole              |JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLife

In [6]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 1470
Number of columns: 35


In [7]:
df.printSchema()

root
 |-- Age: integer (nullable = true)
 |-- Attrition: string (nullable = true)
 |-- BusinessTravel: string (nullable = true)
 |-- DailyRate: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- DistanceFromHome: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- EmployeeCount: integer (nullable = true)
 |-- EmployeeNumber: integer (nullable = true)
 |-- EnvironmentSatisfaction: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HourlyRate: integer (nullable = true)
 |-- JobInvolvement: integer (nullable = true)
 |-- JobLevel: integer (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobSatisfaction: integer (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- MonthlyIncome: integer (nullable = true)
 |-- MonthlyRate: integer (nullable = true)
 |-- NumCompaniesWorked: integer (nullable = true)
 |-- Over18: string (nullable = true)
 |-- OverTime: string 

# Preprocessing and Data Quality Check 

In [11]:
from pyspark.sql.functions import col, sum

missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show()


+---+---------+--------------+---------+----------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+-------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|BusinessTravel|DailyRate|Department|DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|JobRole|JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLifeBalance|YearsAtCompany|YearsInCurrentRole|YearsSinceLastPr

In [12]:
df.groupBy("Attrition").count().show()

+---------+-----+
|Attrition|count|
+---------+-----+
|       No| 1233|
|      Yes|  237|
+---------+-----+



In [13]:
df_clean = df.drop(
    "EmployeeCount",
    "StandardHours",
    "Over18"
)

print("Columns after cleaning:", len(df_clean.columns))

Columns after cleaning: 32


In [14]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "label",
    when(col("Attrition") == "Yes", 1.0).otherwise(0.0)
)

In [15]:
df_clean.select("Attrition", "label").show(10)

+---------+-----+
|Attrition|label|
+---------+-----+
|      Yes|  1.0|
|       No|  0.0|
|      Yes|  1.0|
|       No|  0.0|
|       No|  0.0|
|       No|  0.0|
|       No|  0.0|
|       No|  0.0|
|       No|  0.0|
|       No|  0.0|
+---------+-----+
only showing top 10 rows



In [16]:
df_clean.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0| 1233|
|  1.0|  237|
+-----+-----+



In [17]:
categorical_cols = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime"
]

numeric_cols = [
    "Age",
    "DailyRate",
    "DistanceFromHome",
    "Education",
    "EnvironmentSatisfaction",
    "HourlyRate",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "MonthlyIncome",
    "MonthlyRate",
    "NumCompaniesWorked",
    "PercentSalaryHike",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TotalWorkingYears",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numeric_cols))

Categorical features: 7
Numerical features: 23


In [18]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

In [19]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=[c + "_index" for c in categorical_cols],
    outputCols=[c + "_vec" for c in categorical_cols]
)

In [20]:
from pyspark.ml.feature import VectorAssembler

assembler_inputs = (
    numeric_cols +
    [c + "_vec" for c in categorical_cols]
)

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

In [21]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
)

In [22]:
preprocessing_model = pipeline.fit(df_clean)

In [23]:
df_prepared = preprocessing_model.transform(df_clean)

In [24]:
df_prepared.select(
    "EmployeeNumber",
    "Attrition",
    "label",
    "features"
).show(5, truncate=False)

+--------------+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|EmployeeNumber|Attrition|label|features                                                                                                                                                                                                                      |
+--------------+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1             |Yes      |1.0  |(51,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,16,18,19,20,22,23,27,29,36,37,47,50],[41.0,1102.0,1.0,2.0,2.0,94.0,3.0,2.0,4.0,5993.0,19479.0,8.0,11.0,3.0,1.0,8.0,1.0,6.0,4.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])

In [25]:
ml_data = df_prepared.select(
    "EmployeeNumber",
    "label",
    "features"
)

In [26]:
ml_data.show(5, truncate=False)

+--------------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|EmployeeNumber|label|features                                                                                                                                                                                                                      |
+--------------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1             |1.0  |(51,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,16,18,19,20,22,23,27,29,36,37,47,50],[41.0,1102.0,1.0,2.0,2.0,94.0,3.0,2.0,4.0,5993.0,19479.0,8.0,11.0,3.0,1.0,8.0,1.0,6.0,4.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                      |
|2             |

# Modelling

In [27]:
train_data, test_data = ml_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

Training records: 1216
Testing records: 254


In [28]:
print("Training distribution:")
train_data.groupBy("label").count().orderBy("label").show()

print("Testing distribution:")
test_data.groupBy("label").count().orderBy("label").show()

Training distribution:
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 1021|
|  1.0|  195|
+-----+-----+

Testing distribution:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  212|
|  1.0|   42|
+-----+-----+



# Logistic Regression

In [29]:
# Modelling 

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    probabilityCol="probability",
    predictionCol="prediction"
)

lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

In [30]:
lr_predictions.select(
    "EmployeeNumber",
    "label",
    "probability",
    "prediction"
).show(10, truncate=False)

+--------------+-----+-----------------------------------------+----------+
|EmployeeNumber|label|probability                              |prediction|
+--------------+-----+-----------------------------------------+----------+
|4             |1.0  |[0.2998967341352278,0.7001032658647721]  |1.0       |
|10            |0.0  |[0.7961064476215772,0.20389355237842277] |0.0       |
|12            |0.0  |[0.9235230597687069,0.07647694023129314] |0.0       |
|18            |0.0  |[0.9550661600033622,0.0449338399966378]  |0.0       |
|24            |0.0  |[0.7381807652220049,0.26181923477799507] |0.0       |
|30            |0.0  |[0.9473787518558013,0.05262124814419866] |0.0       |
|38            |0.0  |[0.9040049019399897,0.09599509806001028] |0.0       |
|46            |0.0  |[0.9985117620631919,0.001488237936808079]|0.0       |
|58            |1.0  |[0.9936436244706266,0.006356375529373404]|0.0       |
|60            |0.0  |[0.8508292274963085,0.14917077250369148] |0.0       |
+-----------

# Evaluation Metrics 

In [31]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)

print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.8740157480314961


In [32]:
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

lr_f1 = f1_evaluator.evaluate(lr_predictions)

print("Logistic Regression F1:", lr_f1)

Logistic Regression F1: 0.863430769783914


In [33]:
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

lr_precision = precision_evaluator.evaluate(lr_predictions)
lr_recall = recall_evaluator.evaluate(lr_predictions)

print("Precision:", lr_precision)
print("Recall:", lr_recall)

Precision: 0.8619085777994564
Recall: 0.8740157480314961


In [34]:
lr_predictions.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  203|
|  0.0|       1.0|    9|
|  1.0|       0.0|   23|
|  1.0|       1.0|   19|
+-----+----------+-----+



In [35]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

lr_auc = auc_evaluator.evaluate(lr_predictions)

print("Logistic Regression ROC-AUC:", lr_auc)

Logistic Regression ROC-AUC: 0.8700584007187773


In [36]:
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": lr_accuracy,
    "Precision": lr_precision,
    "Recall": lr_recall,
    "F1": lr_f1,
    "ROC_AUC": lr_auc
}

lr_results

{'Model': 'Logistic Regression',
 'Accuracy': 0.8740157480314961,
 'Precision': 0.8619085777994564,
 'Recall': 0.8740157480314961,
 'F1': 0.863430769783914,
 'ROC_AUC': 0.8700584007187773}

# Random Forest 

In [37]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_data)

rf_predictions = rf_model.transform(test_data)

In [38]:
rf_predictions.select(
    "EmployeeNumber",
    "label",
    "probability",
    "prediction"
).show(10, truncate=False)

+--------------+-----+----------------------------------------+----------+
|EmployeeNumber|label|probability                             |prediction|
+--------------+-----+----------------------------------------+----------+
|4             |1.0  |[0.5492742099243918,0.45072579007560826]|0.0       |
|10            |0.0  |[0.7076239202624199,0.29237607973758]   |0.0       |
|12            |0.0  |[0.9127806294952346,0.08721937050476537]|0.0       |
|18            |0.0  |[0.9058955280902731,0.09410447190972696]|0.0       |
|24            |0.0  |[0.7644992213601475,0.23550077863985247]|0.0       |
|30            |0.0  |[0.6584249983731849,0.3415750016268151] |0.0       |
|38            |0.0  |[0.8658151647744605,0.1341848352255394] |0.0       |
|46            |0.0  |[0.9169742844485301,0.0830257155514699] |0.0       |
|58            |1.0  |[0.9185406191871429,0.08145938081285709]|0.0       |
|60            |0.0  |[0.8996315181772323,0.10036848182276768]|0.0       |
+--------------+-----+---

In [39]:
rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)
rf_precision = precision_evaluator.evaluate(rf_predictions)
rf_recall = recall_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)
rf_auc = auc_evaluator.evaluate(rf_predictions)

print("Random Forest Accuracy :", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall   :", rf_recall)
print("Random Forest F1       :", rf_f1)
print("Random Forest ROC-AUC  :", rf_auc)

Random Forest Accuracy : 0.84251968503937
Random Forest Precision: 0.816992633985268
Random Forest Recall   : 0.84251968503937
Random Forest F1       : 0.7896268401232455
Random Forest ROC-AUC  : 0.7917789757412415


In [40]:
rf_predictions.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  210|
|  0.0|       1.0|    2|
|  1.0|       0.0|   38|
|  1.0|       1.0|    4|
+-----+----------+-----+



In [42]:
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": 0.8740157480314961,
    "Precision": 0.8619085777994564,
    "Recall": 0.8740157480314961,
    "F1": 0.863430769783914,
    "ROC_AUC": 0.8700584007187773
}

In [43]:
rf_results = {
    "Model": "Random Forest",
    "Accuracy": rf_accuracy,
    "Precision": rf_precision,
    "Recall": rf_recall,
    "F1": rf_f1,
    "ROC_AUC": rf_auc
}

In [44]:
import pandas as pd

comparison = pd.DataFrame([
    lr_results,
    rf_results
])

comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.874016,0.861909,0.874016,0.863431,0.870058
1,Random Forest,0.842520,0.816993,0.842520,0.789627,0.791779


In [45]:
final_model = lr_model
final_predictions = lr_predictions

In [47]:
from pyspark.ml.functions import vector_to_array

final_predictions = final_predictions.withColumn(
    "attrition_probability",
    vector_to_array("probability")[1]
)

In [48]:
final_predictions.select(
    "EmployeeNumber",
    "label",
    "prediction",
    "probability",
    "attrition_probability"
).show(10, truncate=False)

+--------------+-----+----------+-----------------------------------------+---------------------+
|EmployeeNumber|label|prediction|probability                              |attrition_probability|
+--------------+-----+----------+-----------------------------------------+---------------------+
|4             |1.0  |1.0       |[0.2998967341352278,0.7001032658647721]  |0.7001032658647721   |
|10            |0.0  |0.0       |[0.7961064476215772,0.20389355237842277] |0.20389355237842277  |
|12            |0.0  |0.0       |[0.9235230597687069,0.07647694023129314] |0.07647694023129314  |
|18            |0.0  |0.0       |[0.9550661600033622,0.0449338399966378]  |0.0449338399966378   |
|24            |0.0  |0.0       |[0.7381807652220049,0.26181923477799507] |0.26181923477799507  |
|30            |0.0  |0.0       |[0.9473787518558013,0.05262124814419866] |0.05262124814419866  |
|38            |0.0  |0.0       |[0.9040049019399897,0.09599509806001028] |0.09599509806001028  |
|46            |0.0 

In [49]:
from pyspark.sql.functions import when

final_predictions = final_predictions.withColumn(
    "risk_level",
    when(col("attrition_probability") < 0.30, "Low")
    .when(col("attrition_probability") < 0.60, "Medium")
    .otherwise("High")
)

In [50]:
final_predictions.select(
    "EmployeeNumber",
    "label",
    "prediction",
    "attrition_probability",
    "risk_level"
).show(20, truncate=False)

+--------------+-----+----------+---------------------+----------+
|EmployeeNumber|label|prediction|attrition_probability|risk_level|
+--------------+-----+----------+---------------------+----------+
|4             |1.0  |1.0       |0.7001032658647721   |High      |
|10            |0.0  |0.0       |0.20389355237842277  |Low       |
|12            |0.0  |0.0       |0.07647694023129314  |Low       |
|18            |0.0  |0.0       |0.0449338399966378   |Low       |
|24            |0.0  |0.0       |0.26181923477799507  |Low       |
|30            |0.0  |0.0       |0.05262124814419866  |Low       |
|38            |0.0  |0.0       |0.09599509806001028  |Low       |
|46            |0.0  |0.0       |0.001488237936808079 |Low       |
|58            |1.0  |0.0       |0.006356375529373404 |Low       |
|60            |0.0  |0.0       |0.14917077250369148  |Low       |
|61            |0.0  |0.0       |0.2731701637004823   |Low       |
|63            |0.0  |0.0       |0.010712945746417568 |Low    

In [51]:
prediction_results = final_predictions.select(
    "EmployeeNumber",
    "label",
    "prediction",
    "attrition_probability",
    "risk_level"
)

prediction_results = prediction_results.withColumnRenamed(
    "label",
    "actual_attrition"
)

In [52]:
prediction_results.show(10, truncate=False)

+--------------+----------------+----------+---------------------+----------+
|EmployeeNumber|actual_attrition|prediction|attrition_probability|risk_level|
+--------------+----------------+----------+---------------------+----------+
|4             |1.0             |1.0       |0.7001032658647721   |High      |
|10            |0.0             |0.0       |0.20389355237842277  |Low       |
|12            |0.0             |0.0       |0.07647694023129314  |Low       |
|18            |0.0             |0.0       |0.0449338399966378   |Low       |
|24            |0.0             |0.0       |0.26181923477799507  |Low       |
|30            |0.0             |0.0       |0.05262124814419866  |Low       |
|38            |0.0             |0.0       |0.09599509806001028  |Low       |
|46            |0.0             |0.0       |0.001488237936808079 |Low       |
|58            |1.0             |0.0       |0.006356375529373404 |Low       |
|60            |0.0             |0.0       |0.14917077250369148 

In [53]:
prediction_results.groupBy(
    "risk_level"
).count().show()

+----------+-----+
|risk_level|count|
+----------+-----+
|      High|   16|
|       Low|  201|
|    Medium|   37|
+----------+-----+



# Saving Prediction Results 

In [54]:
prediction_results = final_predictions.select(
    "EmployeeNumber",
    "label",
    "prediction",
    "attrition_probability",
    "risk_level"
).withColumnRenamed(
    "label",
    "actual_attrition"
)

In [55]:
output_path = "hdfs://namenode:8020/employee_attrition/results/predictions"

prediction_results.write \
    .mode("overwrite") \
    .parquet(output_path)

In [56]:
saved_predictions = spark.read.parquet(output_path)

print("Prediction records:", saved_predictions.count())

saved_predictions.show(10, truncate=False)

Prediction records: 254
+--------------+----------------+----------+---------------------+----------+
|EmployeeNumber|actual_attrition|prediction|attrition_probability|risk_level|
+--------------+----------------+----------+---------------------+----------+
|4             |1.0             |1.0       |0.7001032658647721   |High      |
|10            |0.0             |0.0       |0.20389355237842277  |Low       |
|12            |0.0             |0.0       |0.07647694023129314  |Low       |
|18            |0.0             |0.0       |0.0449338399966378   |Low       |
|24            |0.0             |0.0       |0.26181923477799507  |Low       |
|30            |0.0             |0.0       |0.05262124814419866  |Low       |
|38            |0.0             |0.0       |0.09599509806001028  |Low       |
|46            |0.0             |0.0       |0.001488237936808079 |Low       |
|58            |1.0             |0.0       |0.006356375529373404 |Low       |
|60            |0.0             |0.0    

In [57]:
# Generate HBase PUT commands from Spark prediction results

hbase_cmds = []

for row in prediction_results.collect():
    emp = str(row["EmployeeNumber"])
    prob = str(row["attrition_probability"])
    pred = str(int(row["prediction"]))
    risk = row["risk_level"]

    hbase_cmds.append(
        f"put 'employee_attrition_predictions', '{emp}', "
        f"'prediction:probability', '{prob}'"
    )

    hbase_cmds.append(
        f"put 'employee_attrition_predictions', '{emp}', "
        f"'prediction:prediction', '{pred}'"
    )

    hbase_cmds.append(
        f"put 'employee_attrition_predictions', '{emp}', "
        f"'prediction:risk_level', '{risk}'"
    )

    hbase_cmds.append(
        f"put 'employee_attrition_predictions', '{emp}', "
        f"'prediction:model_version', 'LogisticRegression'"
    )

# Save commands inside Jupyter container
with open("/home/jovyan/hbase_puts.txt", "w") as f:
    f.write("\n".join(hbase_cmds))

print("Employees:", prediction_results.count())
print("HBase PUT commands:", len(hbase_cmds))
print("File created: /home/jovyan/hbase_puts.txt")

Employees: 254
HBase PUT commands: 1016
File created: /home/jovyan/hbase_puts.txt
